# 02 - Exploratory Data Analysis Overview
High-level EDA of the master panel: distributions, correlations, temporal patterns.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from hdi.config import *
from hdi.data.loaders import load_master_panel
from hdi.visualization.themes import apply_theme
apply_theme()

## 1. Panel Overview

In [ ]:
panel = load_master_panel()
print(f"Shape: {panel.shape}")
print(f"Countries: {panel['iso3'].nunique()}")
print(f"Years: {panel['year'].min()}-{panel['year'].max()}")
print(f"\nColumn types:")
print(panel.dtypes.value_counts())
print(f"\nMissing fraction (top 20):")
print(panel.isnull().mean().sort_values(ascending=False).head(20))

## 2. Key Variable Distributions

In [ ]:
key_vars = ['life_expectancy', 'gdp_pc_ppp', 'health_exp_pct_gdp', 
            'physicians_per_1000', 'hospital_beds_per_1000']
available = [v for v in key_vars if v in panel.columns]

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(4*len(available), 4))
    if len(available) == 1:
        axes = [axes]
    for ax, var in zip(axes, available):
        panel[var].dropna().hist(bins=50, ax=ax, alpha=0.7)
        ax.set_title(var.replace('_', ' ').title(), fontsize=10)
    plt.tight_layout()
    plt.show()

## 3. Temporal Trends by Region

In [ ]:
if 'who_region' in panel.columns and 'life_expectancy' in panel.columns:
    regional = panel.groupby(['year', 'who_region'])['life_expectancy'].mean().reset_index()
    fig, ax = plt.subplots(figsize=(10, 6))
    for region in regional['who_region'].dropna().unique():
        subset = regional[regional['who_region'] == region]
        ax.plot(subset['year'], subset['life_expectancy'], label=region, linewidth=1.5)
    ax.set_xlabel('Year')
    ax.set_ylabel('Life Expectancy')
    ax.set_title('Life Expectancy Trends by WHO Region')
    ax.legend()
    plt.show()

## 4. Correlation Matrix

In [ ]:
numeric_cols = panel.select_dtypes(include='number').columns
if len(numeric_cols) > 2:
    corr = panel[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, ax=ax,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    ax.set_title('Correlation Matrix of Key Variables')
    plt.tight_layout()
    plt.show()

## 5. Income Group Comparisons

In [ ]:
if 'wb_income' in panel.columns:
    for var in ['life_expectancy', 'health_exp_pct_gdp']:
        if var in panel.columns:
            fig, ax = plt.subplots(figsize=(8, 5))
            panel.boxplot(column=var, by='wb_income', ax=ax)
            ax.set_title(f'{var} by Income Group')
            plt.suptitle('')
            plt.show()